# Обучение нейросетевых классификаторов

Обучим небольшую MLP на подготовленных табличных данных. Для CLASS и NSP используем отдельные сети: первая определяет один из десяти паттернов, вторая — одно из трёх состояний. NSP не вычисляется автоматически из CLASS.

За основу берём учебный пример `005_netology_PyTorch_mlp_task.ipynb`: создание сети в PyTorch, явный цикл обучения и сравнение экспериментов. Здесь задача многоклассовая, поэтому изменятся выходной слой и функция потерь. Основная цель — разобраться в обучении сети и сравнить её с классическими моделями. Ниже пока только план, без выполнения экспериментов.

## 1. Загрузка данных и общих настроек

Загрузим train.csv, validation.csv и test.csv из runtime/data/prepared/. Используем те же 21 признак и то же разбиение, что в предыдущем блокноте. Повторно делить данные и выбирать признаки не требуется. Сегменты одного исследования уже находятся в одной части выборки, поэтому сеть не увидит соседние сегменты тестового исследования при обучении.

Загрузим ExperimentSettings из experiments/settings.py, а соответствия меток и ClassificationTarget — из experiments/constants.py. Настройки обучения и отдельное имя эксперимента MLflow добавим в ExperimentSettings. Зафиксируем seed из настроек для NumPy и PyTorch.

Ожидаемый результат: входы и две цели для каждой части выборки; воспроизводимые настройки эксперимента.

## 2. Подготовка данных для PyTorch

Обучим StandardScaler только на train; этим же преобразованием обработаем validation и test. Сохраним scaler вместе с сетью, чтобы сервис обрабатывал входы так же, как при обучении.

Преобразуем признаки в тензоры float32, а целевые метки — в целые индексы: 0–9 для CLASS и 0–2 для NSP. В отчётах вернём исходные коды и названия категорий.

Создадим TensorDataset и DataLoader. Обучающие записи перемешиваем с фиксированным seed; validation и test — без перемешивания. Размер batch зададим в настройках. Начнём с CPU на Mac; ускорение через MPS проверим отдельно, если оно потребуется.

Ожидаемый результат: данные поступают в сеть небольшими batch, а преобразование признаков не использует информацию из validation и test.

## 3. Устройство небольшой MLP

Начнём с одного скрытого слоя: 21 вход → 32 нейрона → выход. Между слоями используем ReLU. Число выходов — 10 для CLASS и 3 для NSP. Покажем архитектуру и количество обучаемых параметров.

Выход сети — logits, то есть оценки классов до преобразования в вероятности. Используем CrossEntropyLoss: она принимает logits и индекс правильного класса. Softmax внутрь сети не добавляем; он понадобится при получении вероятностей для демонстрации. Предсказанный класс выбираем через argmax.

Ожидаемый результат: понятная небольшая сеть и объяснение роли входного, скрытого и выходного слоёв.

## 4. Цикл обучения и базовая MLP

Используем Adam. Для каждого batch явно выполним шаги: обнуление градиентов → прямой проход → расчёт loss → backward → обновление весов. Это основной участок для разбора того, как сеть обучается.

На train включаем model.train(); для проверки — model.eval() и torch.no_grad(). Среднюю loss за эпоху считаем с учётом количества записей в batch. После каждой эпохи оцениваем validation и сохраняем историю loss и macro-F1. Функция потерь управляет обновлением весов, а macro-F1 показывает качество классификации.

Ограничим число эпох и предусмотрим раннюю остановку, если validation macro-F1 не улучшается. Сохраняем веса лучшей эпохи по validation, а не просто последней. Все параметры остановки фиксируем до запуска.

Обучим отдельную базовую MLP для каждой цели с новой инициализацией весов. В MLflow сохраняем настройки, seed, архитектуру, метрики по эпохам и результаты лучшей эпохи.

Ожидаемый результат: два исходных нейросетевых классификатора и история их обучения.

## 5. Разбор обучения и ошибок

Построим кривые train/validation loss и macro-F1. Проверим, обучается ли сеть, когда улучшение прекращается и есть ли переобучение. Если train продолжает улучшаться, а validation ухудшается, сеть хуже переносит выученные закономерности на другие исследования.

В компактной таблице покажем macro-F1, weighted-F1 и accuracy. Основная метрика — macro-F1: каждый класс имеет одинаковый вес. Weighted-F1 — дополнительная метрика, поскольку частые категории могут скрыть слабое качество редких. Если macro-F1 растёт, а weighted-F1 падает, проверим, улучшились ли редкие классы за счёт частых.

Для CLASS и NSP отдельно покажем матрицы ошибок и столбчатые диаграммы precision, recall и F1 по классам. Для NSP отдельно посчитаем Pathologic → Normal и Suspect → Normal: такие ошибки мешают своевременно выделить подозрительные записи.

Ожидаемый результат: выводы о процессе обучения и конкретных ошибках базовой MLP.

## 6. Эксперименты и подбор параметров

Сначала проведём несколько понятных экспериментов, изменяя по одному условию относительно базовой MLP:

- размер скрытого слоя и добавление второго слоя;
- learning rate Adam;
- регуляризация: weight_decay и Dropout;
- веса классов в CrossEntropyLoss, рассчитанные только по train.

Для каждого запуска используем новую сеть, фиксируем настройки и сохраняем результаты в MLflow. Сравним варианты на validation и коротко объясним влияние изменений. Усложнение сети само по себе не считается улучшением.

После разбора этих экспериментов выполним небольшой подбор Optuna с TPESampler, как в примере coffee_ml_pipeline.ipynb: простой objective и явные диапазоны параметров. Подбираем параметры отдельно для CLASS и NSP. Бюджет нейросетевого подбора и диапазоны зададим перед реализацией; число попыток из классического эксперимента автоматически не переносим.

Ожидаемый результат: таблица результатов и вручную выбранные варианты для обеих целей с объяснением выбора.

## 7. Сравнение с классическими моделями на validation

Для каждой цели сравним исходный RandomForest — согласованный baseline, выбранный RandomForest из предыдущего блокнота, базовую MLP и MLP после экспериментов. Сохраним одинаковые данные и определения метрик.

При выборе ориентируемся на macro-F1; при близких результатах учитываем weighted-F1. Проверяем согласованные ограничения относительно исходного RandomForest: weighted-F1 и accuracy не должны снижаться более чем на 0,01, а для NSP число ошибок Pathologic → Normal и Suspect → Normal не должно увеличиваться.

Лучший вариант выбираем вручную по таблице и матрицам ошибок. Для CLASS и NSP могут подойти разные модели. Если MLP уступает лесу, зафиксируем этот результат: задача эксперимента — проверить подход, а не гарантировать победу нейросети.

Ожидаемый результат: обоснованный выбор моделей и зафиксированные параметры до проверки на test.

## 8. Проверка на test и итоговые выводы

Оценим зафиксированные нейросетевые варианты на всех 421 записях test.csv. Сравним их с уже полученными тестовыми результатами исходного и выбранного RandomForest на этих же записях.

Покажем accuracy, macro/weighted-F1, macro-precision и macro-recall, матрицы ошибок и критичные ошибки NSP. Напишем короткие структурированные выводы, вручную сверив все числа с выполненными ячейками.

Test не используем для подбора параметров MLP. Его результаты уже рассматривались в классических экспериментах; это ограничение учитываем при оценке итоговых результатов.

Ожидаемый результат: сравнение качества на общей тестовой выборке и вывод, какой подход выбираем для каждой цели.

## 9. Сохранение моделей для последующего сервиса

Сохраним веса выбранных MLP, описание архитектуры, scaler, порядок входных признаков и соответствие выходных индексов исходным меткам. Рядом сохраним настройки и итоговые метрики; модели и результаты зарегистрируем в MLflow. Файлы разместим в runtime/models/neural/.

Проверим загрузку сохранённых моделей: заново создадим сеть, загрузим веса и scaler и убедимся, что предсказания совпадают с результатами до сохранения. Обучение при загрузке не выполняется.

После сравнения классических и нейросетевых моделей вынесем выбранные шаги подготовки, обучения и сохранения в воспроизводимый пайплайн по примеру coffee_ml_pipeline_dag.py. Управление останется через just и общие настройки проекта. Реализация пайплайна, API и интерфейса — следующий этап.

Ожидаемый результат: артефакты, которые можно загрузить в сервис без повторного обучения.